In [4]:
import numpy as np
from qibo import models, gates

from tncdr.stabilizer_mps.tensor_network import TensorNetwork

# Import W tensor generator
from tncdr.stabilizer_mps.w_utils import _compute_all_w_tensors

# Import 'spatial' measurements (i.e. initial state and observable)
from tncdr.stabilizer_mps.w_utils import pauli_pauli_expansion, basis_pauli_expansion
# Import 'temporal' measurements (i.e. theta and X fictitous states)
from tncdr.stabilizer_mps.w_utils import  X_pauli_expansion, theta_pauli_expansion

Ws = _compute_all_w_tensors()

def sample_random_pauli(n:int):
    possible_symbols = list('IXYZ')
    return ''.join([possible_symbols[np.random.randint(len(possible_symbols))] for _ in range(n)])

def pauli_tensors(pauli_string:str):
    for p in pauli_string:
        if 'IXYZ'.find(p) >= 0:
            yield Ws[p]

/home/matteo/Documents/PhD/tncdr/src/tncdr/stabilizer_mps/w_utils.py:92: ComplexWarning: Casting complex values to real discards the imaginary part
  return tn.tensornet.nodes['F']['tensor'].astype(float)/4


In [55]:
N=3
    
rotation_pauli = sample_random_pauli(N)
observable = "XXX"
theta =  2*np.pi*np.random.rand()
intial_state = '0' * N

print(f'Rotation generator: {rotation_pauli}')
print(f'H: {observable}')
print(f'rho_0: |{intial_state}X{intial_state}|')
print(f'theta: {theta}, {1-2*np.sin(theta/2)**2}, {2*np.sin(theta/2)*np.cos(theta/2)}')

# Build TensorNetwork
tn = TensorNetwork()

# Add initial state
for n, state in enumerate(intial_state):
    tn.add_tensor(f'T{n}', tensor=basis_pauli_expansion(state))

# Add one rotation
for n, w_tensor in enumerate(pauli_tensors(rotation_pauli)):
    tn.add_tensor(id=f'W{n}', tensor=w_tensor)
    tn.add_edge(f'T{n}', f'W{n}', 'v_link', (0,2))

tn.add_tensor(id='theta', tensor=theta_pauli_expansion(theta=theta))
tn.add_edge('theta', 'W0', 'h_link', (0,1))

for n in range(N-1):
    tn.add_edge(f'W{n}', f'W{n+1}', 'h_link', (0,1))

tn.add_tensor(id='X', tensor=X_pauli_expansion())
tn.add_edge(f'W{N-1}', 'X', 'h_link', (0,0))

# Connect to the observable
for n, o in enumerate(observable):
    tn.add_tensor(f'O{n}', tensor=pauli_pauli_expansion(o))
    tn.add_edge(f'O{n}', f'W{n}', 'v_link', (0,3))

# Contract the TensorNetwork
tn.contract('theta', 'W0', 'h_link', 'tmp')
for n in range(N-1):
    tn.contract(f'T{n}', 'tmp', 'v_link', 'tmp2')
    tn.contract(f'O{n}', 'tmp2', 'v_link', 'tmp3')
    tn.contract(f'tmp3', f'W{n+1}', 'h_link', 'tmp')

tn.contract(f'T{N-1}', 'tmp', 'v_link', 'tmp2')
tn.contract(f'O{N-1}', 'tmp2', 'v_link', 'tmp3')
tn.contract('tmp3', 'X', 'h_link', 'F')

# Output the result
print(tn.tensornet.nodes['F']['tensor'].item())

Rotation generator: YXX
H: XXX
rho_0: |000X000|
theta: 3.5653041340495375, -0.9115692599003742, -0.41114654857445193
0.4111465485744519


In [24]:
print("Rotation Pauli string:", rotation_pauli)
print("Observable Pauli string:", observable)
print("Theta:", theta)

# Create a Qibo circuit with N qubits.
circuit = models.Circuit(N)

# --- State Preparation ---
# Prepare each qubit in |1> by applying an X gate (since |1> = X|0>)
for i in range(N):
    circuit.add(gates.X(i))

# --- Parameterized Rotation ---
circuit.add(gates.RY(0, theta))

# --- Entangling Operations ---
# To mimic the horizontal connections (h_links) that “chain” the W tensors,
# we apply CNOT gates between neighboring qubits.
for i in range(N - 1):
    circuit.add(gates.CNOT(i, i + 1))

# --- Final Projection ---
# In the tensor network, the final contraction applies an X projection.
# Here we transform the last qubit to the X-basis by applying a Hadamard.
circuit.add(gates.H(N - 1))

# --- Observable Basis Change ---
# For each qubit, we now change the measurement basis to that of the chosen observable.
# In our simulation, the observable for each qubit is a Pauli operator.
# To measure in the X basis, we apply H; for Y, we apply S† then H; for Z (or I), no change is needed.
for i, p in enumerate(observable):
    if p == "X":
        # For qubits not already in the X basis, apply H.
        if i != N - 1:
            circuit.add(gates.H(i))
    elif p == "Y":
        # To measure in the Y basis, apply S† (phase gate dagger) then H.
        circuit.add(gates.S(i).dagger())
        circuit.add(gates.H(i))
    # For 'Z' or 'I', we assume measurement in the computational (Z) basis.

# --- Measurement ---
# Finally, measure all qubits.
circuit.add(gates.M(*range(N)))

# Print out the circuit for verification.
print(circuit)

# Optionally, run the circuit on a simulator to get measurement outcomes:
result = circuit()
print("Measurement outcomes:", result)

Rotation Pauli string: XYI
Observable Pauli string: YYI
Theta: 2.659338986428602
0: ─X─RY─o───SDG─H─M─
1: ─X────X─o─SDG─H─M─
2: ─X──────X─H─────M─
Measurement outcomes: 0.2589j|000> + 0.42775j|001> + -0.42775j|010> + -0.2589j|011> + 0.42775j|100> + 0.2589j|101> + -0.2589j|110> + -0.42775j|111>


In [25]:
result.state()

array([0.+0.25889742j, 0.+0.42775241j, 0.-0.42775241j, 0.-0.25889742j,
       0.+0.42775241j, 0.+0.25889742j, 0.-0.25889742j, 0.-0.42775241j])

In [26]:
from qibo import hamiltonians
from qibo.symbols import X, Y, Z, I

In [27]:
form = Y(0) + Y(1) + I(2)
ham = hamiltonians.SymbolicHamiltonian(form)

In [28]:
ham.expectation(result.state())

0.9999999999999993